In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install pdfplumber tqdm
import os

PDF_FOLDER = "/content/drive/MyDrive/Narkotika"

pdf_files = [
    f for f in os.listdir(PDF_FOLDER)
    if f.endswith(".pdf")
]

print("Jumlah PDF:", len(pdf_files))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 60.0 MB/s eta 0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatible.
Jumlah PDF: 50


In [3]:
import os

OUTPUT_FOLDER = "/content/data/raw"
LOG_FOLDER = "/content/logs"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(LOG_FOLDER, exist_ok=True)

In [8]:
import pdfplumber
import re
import os
from tqdm import tqdm

PDF_FOLDER = "/content/drive/MyDrive/Narkotika"
OUTPUT_FOLDER = "/content/data/raw"

def extract_text(pdf_path):

    text = ""

    with pdfplumber.open(pdf_path) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text(
                x_tolerance=3,
                y_tolerance=3
            )

            if not page_text:
                continue

            lines = page_text.split("\n")

            clean_lines = []

            for line in lines:

                line = line.strip()

                # buang disclaimer
                if "disclaimer kepaniteraan" in line.lower():
                    continue

                if "mahkamah agung republik indonesia" in line.lower():
                    continue

                if "putusan.mahkamahagung.go.id" in line.lower():
                    continue

                # buang baris yang didominasi huruf tunggal
                words = line.split()

                if len(words) >= 3:

                    single_words = sum(
                        1 for w in words
                        if len(w) == 1
                    )

                    ratio = single_words / len(words)

                    if ratio > 0.5:
                        continue

                clean_lines.append(line)

            text += "\n".join(clean_lines) + "\n"

    return text


import re

def clean_text(text):

    text = text.lower()

    # hapus blok disclaimer panjang
    text = re.sub(
        r'disclaimer.*?hubungi kepaniteraan mahkamah agung ri melalui',
        ' ',
        text,
        flags=re.DOTALL
    )

    # hapus sisa watermark
    text = re.sub(
        r'\b[a-z]\b',
        ' ',
        text
    )

    # hapus karakter aneh
    text = re.sub(
        r'[^a-z0-9\s]',
        ' ',
        text
    )

    # rapikan spasi
    text = re.sub(
        r'\s+',
        ' ',
        text
    )



    return text.strip()

In [9]:
pdf_files = sorted([
    f for f in os.listdir(PDF_FOLDER)
    if f.endswith(".pdf")
])

test_pdf = os.path.join(
    PDF_FOLDER,
    pdf_files[0]
)

raw_text = extract_text(test_pdf)

cleaned = clean_text(raw_text)

print(cleaned[:3000])

gnomor 102 pid sus 2025 pn tlg demi keadilan berdasarkan ketuhanan yang maha esa pengadilan negeri tulungagung yang mengadili perkara pidana dengan acara pemeriksaan biasa dalam tingkat pertama menjatuhkan aputusan sebagai berikut dalam perkara terdakwa 1 nama lengkap adila resti sari binti boimain h2 tempat lahir blitar 3 umur tanggal lahir 26 tahun 01 januiari 1999 4 jenis kelamin perempuan 5 kebangsaan indonesia 6 tempat tinggal sesuai ktp dsn ampelgading rt 02 rw 03 ds ampilgading kec selorejo kab blitar domisili erumah kos masuk lingkungan 10 ds kec rngunut kab tulungagung 7 agama islam 8 pekerjaan mengurus rumah tangga terdakwa ditangkap sejak 6 februari 2025 terdakwa ditahan dalam tahanan rumah tahanan negara oleh 1 penyidik sejak tanggal 7 februari 2025 sampai dengan tanggal 26 februari 2025 a2 penyidik perpanjangan oleh penuntut umum sejak tanggal 27 februari 2025 sampai dengan tanggal 7 april 2025 h3 penyidik perpanjangan pertama oleh ketua penkgadilan negeri sejak tanggal 8 

In [10]:
log = []

pdf_files = sorted([
    f for f in os.listdir(PDF_FOLDER)
    if f.endswith(".pdf")
])

for idx, file in enumerate(tqdm(pdf_files), start=1):

    pdf_path = os.path.join(
        PDF_FOLDER,
        file
    )

    raw_text = extract_text(pdf_path)

    cleaned = clean_text(raw_text)

    txt_name = f"case_{idx:03d}.txt"

    output_path = os.path.join(
        OUTPUT_FOLDER,
        txt_name
    )

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(cleaned)

    log.append(
        f"{file} -> {txt_name} | words={len(cleaned.split())}"
    )

print("Selesai")

100%|██████████| 50/50 [06:24<00:00,  7.69s/it]

Selesai


In [11]:
with open(
    "/content/data/raw/case_001.txt",
    "r",
    encoding="utf-8"
) as f:

    text = f.read()

print(text[:3000])

gnomor 102 pid sus 2025 pn tlg demi keadilan berdasarkan ketuhanan yang maha esa pengadilan negeri tulungagung yang mengadili perkara pidana dengan acara pemeriksaan biasa dalam tingkat pertama menjatuhkan aputusan sebagai berikut dalam perkara terdakwa 1 nama lengkap adila resti sari binti boimain h2 tempat lahir blitar 3 umur tanggal lahir 26 tahun 01 januiari 1999 4 jenis kelamin perempuan 5 kebangsaan indonesia 6 tempat tinggal sesuai ktp dsn ampelgading rt 02 rw 03 ds ampilgading kec selorejo kab blitar domisili erumah kos masuk lingkungan 10 ds kec rngunut kab tulungagung 7 agama islam 8 pekerjaan mengurus rumah tangga terdakwa ditangkap sejak 6 februari 2025 terdakwa ditahan dalam tahanan rumah tahanan negara oleh 1 penyidik sejak tanggal 7 februari 2025 sampai dengan tanggal 26 februari 2025 a2 penyidik perpanjangan oleh penuntut umum sejak tanggal 27 februari 2025 sampai dengan tanggal 7 april 2025 h3 penyidik perpanjangan pertama oleh ketua penkgadilan negeri sejak tanggal 8 

In [17]:
with open(
    "/content/logs/cleaning.log",
    "w",
    encoding="utf-8"
) as f:

    for item in log:
        f.write(item + "\n")

print("Log berhasil dibuat")

Log berhasil dibuat


In [12]:
!git config --global user.email "aschyprdn@gmail.com"
!git config --global user.name "AsyaPradana"

In [13]:
import os

folders = [
    "CBR-Narkotika-PN-Tulungagung/data/raw",
    "CBR-Narkotika-PN-Tulungagung/data/processed",
    "CBR-Narkotika-PN-Tulungagung/logs",
    "CBR-Narkotika-PN-Tulungagung/notebooks",
    "CBR-Narkotika-PN-Tulungagung/src"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Struktur folder berhasil dibuat")

Struktur folder berhasil dibuat


In [14]:
import shutil

shutil.copytree(
    "/content/data/raw",
    "/content/CBR-Narkotika-PN-Tulungagung/data/raw",
    dirs_exist_ok=True
)

print("File TXT berhasil dipindahkan")

File TXT berhasil dipindahkan


In [18]:
import shutil

shutil.copy(
    "/content/logs/cleaning.log",
    "/content/CBR-Narkotika-PN-Tulungagung/logs/cleaning.log"
)

'/content/CBR-Narkotika-PN-Tulungagung/logs/cleaning.log'

In [19]:
requirements = """
pdfplumber
pandas
numpy
scikit-learn
nltk
tqdm
"""

with open(
    "/content/CBR-Narkotika-PN-Tulungagung/requirements.txt",
    "w"
) as f:
    f.write(requirements)

print("requirements.txt berhasil dibuat")

requirements.txt berhasil dibuat


In [20]:
gitignore = """
__pycache__/
.ipynb_checkpoints/
*.zip
"""

with open(
    "/content/CBR-Narkotika-PN-Tulungagung/.gitignore",
    "w"
) as f:
    f.write(gitignore)

print(".gitignore berhasil dibuat")

.gitignore berhasil dibuat


In [21]:
readme = """
# CBR Narkotika PN Tulungagung

## Deskripsi

Implementasi Case-Based Reasoning (CBR) untuk analisis putusan perkara narkotika pada Pengadilan Negeri Tulungagung.

## Dataset

50 putusan narkotika yang diperoleh dari Direktori Putusan Mahkamah Agung Republik Indonesia.

## Tahap 1 - Membangun Case Base

- Seleksi dan unduh putusan
- Konversi PDF ke TXT
- Cleaning teks
- Validasi hasil ekstraksi
- Penyimpanan corpus

## Struktur Folder

data/raw : hasil ekstraksi teks putusan

logs : log cleaning

notebooks : notebook Google Colab

src : source code Python
"""

with open(
    "/content/CBR-Narkotika-PN-Tulungagung/README.md",
    "w"
) as f:
    f.write(readme)

print("README.md berhasil dibuat")

README.md berhasil dibuat


In [33]:
!ls -la /content

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
total 28
drwxr-xr-x 1 root root 4096 Jun 15 05:36 .
drwxr-xr-x 1 root root 4096 Jun 15 05:02 ..
drwxr-xr-x 4 root root 4096 Jun  4 13:32 .config
drwxr-xr-x 3 root root 4096 Jun 15 05:12 data
drwx------ 5 root root 4096 Jun 15 05:12 drive
drwxr-xr-x 2 root root 4096 Jun 15 05:30 logs
drwxr-xr-x 1 root root 4096 Jun  4 13:32 sample_data


In [39]:
import os

folders = [
    "data/raw",
    "data/processed",
    "logs",
    "notebooks",
    "src"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

open("data/processed/.gitkeep", "w").close()

print("Struktur repository berhasil dibuat")

Struktur repository berhasil dibuat


In [40]:
import shutil

shutil.copytree(
    "/content/data/raw",
    "/content/CBR-Narkotika-PN-Tulungagung/data/raw",
    dirs_exist_ok=True
)

print("50 file TXT berhasil dipindahkan")

50 file TXT berhasil dipindahkan


In [41]:
shutil.copy(
    "/content/logs/cleaning.log",
    "/content/CBR-Narkotika-PN-Tulungagung/logs/cleaning.log"
)

'/content/CBR-Narkotika-PN-Tulungagung/logs/cleaning.log'

In [42]:
%%writefile /content/CBR-Narkotika-PN-Tulungagung/requirements.txt
pdfplumber
pandas
numpy
scikit-learn
nltk
tqdm

Writing /content/CBR-Narkotika-PN-Tulungagung/requirements.txt


In [43]:
%%writefile /content/CBR-Narkotika-PN-Tulungagung/.gitignore
__pycache__/
.ipynb_checkpoints/
*.zip

Writing /content/CBR-Narkotika-PN-Tulungagung/.gitignore


In [44]:
!find /content/CBR-Narkotika-PN-Tulungagung -type f | sort

/content/CBR-Narkotika-PN-Tulungagung/data/processed/.gitkeep
/content/CBR-Narkotika-PN-Tulungagung/data/raw/case_001.txt
/content/CBR-Narkotika-PN-Tulungagung/data/raw/case_002.txt
/content/CBR-Narkotika-PN-Tulungagung/data/raw/case_003.txt
/content/CBR-Narkotika-PN-Tulungagung/data/raw/case_004.txt
/content/CBR-Narkotika-PN-Tulungagung/data/raw/case_005.txt
/content/CBR-Narkotika-PN-Tulungagung/data/raw/case_006.txt
/content/CBR-Narkotika-PN-Tulungagung/data/raw/case_007.txt
/content/CBR-Narkotika-PN-Tulungagung/data/raw/case_008.txt
/content/CBR-Narkotika-PN-Tulungagung/data/raw/case_009.txt
/content/CBR-Narkotika-PN-Tulungagung/data/raw/case_010.txt
/content/CBR-Narkotika-PN-Tulungagung/data/raw/case_011.txt
/content/CBR-Narkotika-PN-Tulungagung/data/raw/case_012.txt
/content/CBR-Narkotika-PN-Tulungagung/data/raw/case_013.txt
/content/CBR-Narkotika-PN-Tulungagung/data/raw/case_014.txt
/content/CBR-Narkotika-PN-Tulungagung/data/raw/case_015.txt
/content/CBR-Narkotika-PN-Tulungagung/

In [45]:
%cd /content/CBR-Narkotika-PN-Tulungagung

!git status

/content/CBR-Narkotika-PN-Tulungagung
On branch Modul2
Your branch is up to date with 'origin/Modul2'.

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore
	data/
	logs/
	requirements.txt

nothing added to commit but untracked files present (use "git add" to track)


In [46]:
!git add .

In [47]:
!git status

On branch Modul2
Your branch is up to date with 'origin/Modul2'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   .gitignore
	new file:   data/processed/.gitkeep
	new file:   data/raw/case_001.txt
	new file:   data/raw/case_002.txt
	new file:   data/raw/case_003.txt
	new file:   data/raw/case_004.txt
	new file:   data/raw/case_005.txt
	new file:   data/raw/case_006.txt
	new file:   data/raw/case_007.txt
	new file:   data/raw/case_008.txt
	new file:   data/raw/case_009.txt
	new file:   data/raw/case_010.txt
	new file:   data/raw/case_011.txt
	new file:   data/raw/case_012.txt
	new file:   data/raw/case_013.txt
	new file:   data/raw/case_014.txt
	new file:   data/raw/case_015.txt
	new file:   data/raw/case_016.txt
	new file:   data/raw/case_017.txt
	new file:   data/raw/case_018.txt
	new file:   data/raw/case_019.txt
	new file:   data/raw/case_020.txt
	new file:   data/raw/case_021.txt
	new file:   data/raw/case_022.txt
	new file:   data/raw/case

In [48]:
!git commit -m "Tahap 1 - Membangun Case Based"

[Modul2 cc973cc] Tahap 1 - Membangun Case Based
 54 files changed, 109 insertions(+)
 create mode 100644 .gitignore
 create mode 100644 data/processed/.gitkeep
 create mode 100644 data/raw/case_001.txt
 create mode 100644 data/raw/case_002.txt
 create mode 100644 data/raw/case_003.txt
 create mode 100644 data/raw/case_004.txt
 create mode 100644 data/raw/case_005.txt
 create mode 100644 data/raw/case_006.txt
 create mode 100644 data/raw/case_007.txt
 create mode 100644 data/raw/case_008.txt
 create mode 100644 data/raw/case_009.txt
 create mode 100644 data/raw/case_010.txt
 create mode 100644 data/raw/case_011.txt
 create mode 100644 data/raw/case_012.txt
 create mode 100644 data/raw/case_013.txt
 create mode 100644 data/raw/case_014.txt
 create mode 100644 data/raw/case_015.txt
 create mode 100644 data/raw/case_016.txt
 create mode 100644 data/raw/case_017.txt
 create mode 100644 data/raw/case_018.txt
 create mode 100644 data/raw/case_019.txt
 create mode 100644 data/raw/case_020.txt


In [52]:
!git push origin Modul2

Enumerating objects: 61, done.
Counting objects: 100% (61/61), done.
Delta compression using up to 2 threads
Compressing objects: 100% (55/55), done.
Writing objects: 100% (60/60), 563.85 KiB | 2.44 MiB/s, done.
Total 60 (delta 11), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (11/11), done.
To https://github.com/AsyaPradana/CBR-Narkotika-PN-Tulungagung.git
   859e423..cc973cc  Modul2 -> Modul2


In [50]:
!git remote -v

origin	https://github.com/AsyaPradana/CBR-Narkotika-PN-Tulungagung.git (fetch)
origin	https://github.com/AsyaPradana/CBR-Narkotika-PN-Tulungagung.git (push)


In [54]:
import os

os.makedirs(
    "/content/CBR-Narkotika-PN-Tulungagung/notebooks",
    exist_ok=True
)

print("Folder notebooks berhasil dibuat")

Folder notebooks berhasil dibuat


In [57]:
import shutil

shutil.move(
    "/content/Tahap1_CaseBased.ipynb",
    "/content/CBR-Narkotika-PN-Tulungagung/notebooks/Tahap1_CaseBased.ipynb"
)

print("Notebook berhasil dipindahkan")

Notebook berhasil dipindahkan


In [68]:
!git status

On branch Modul2
Your branch is up to date with 'origin/Modul2'.

Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   notebooks/Tahap1_CaseBased.ipynb



In [75]:
!git commit --amend --no-edit

[Modul2 08d56b8] notebook Tahap 1
 Date: Mon Jun 15 06:16:08 2026 +0000
 1 file changed, 1219 insertions(+)
 create mode 100644 notebooks/Tahap1_CaseBased.ipynb


In [79]:
%cd /content/CBR-Narkotika-PN-Tulungagung

git add notebooks/Tahap1_CaseBased.ipynb
git commit --amend --no-edit

SyntaxError: invalid syntax (1532612680.py, line 3)